# ResNet34 Stratified 5-Fold Cross Validation

## Objective

This notebook evaluates ResNet34 using Stratified 5-Fold Cross Validation for the screen recapture detection task.

ResNet34 provides a deeper residual architecture than ResNet18, enabling the model to learn richer visual representations while maintaining efficient inference.

The objective is to compare its performance against all previously evaluated classical and deep learning models and identify the most suitable architecture for deployment.

## Import Required Libraries

Import the libraries required for transfer learning, dataset loading, optimization, and evaluation.

In [1]:
import os
import copy
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

## Image Preprocessing

Define separate preprocessing pipelines for training and validation using ImageNet normalization and data augmentation.

In [2]:
train_transform = transforms.Compose([

    transforms.Resize((256,256)),

    transforms.RandomCrop(224),

    transforms.RandomHorizontalFlip(0.5),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

test_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [3]:
dataset_path = "../dataset"

dataset = datasets.ImageFolder(dataset_path)

labels = dataset.targets

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cpu


## Training Utilities

Implement reusable training and evaluation functions used throughout the cross-validation process.

In [4]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs,1)

        total += labels.size(0)
        correct += (preds==labels).sum().item()

    return running_loss/len(loader), correct/total


def evaluate(model, loader):

    model.eval()

    predictions = []
    targets = []

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs,1)

            predictions.extend(preds.cpu().numpy())
            targets.extend(labels.cpu().numpy())

            total += labels.size(0)
            correct += (preds==labels).sum().item()

    accuracy = correct/total

    return accuracy,predictions,targets

In [5]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

best_fold_accuracy = 0

best_fold = -1

os.makedirs("../models",exist_ok=True)

EPOCHS = 40

PATIENCE = 5

## Train ResNet34

Fine-tune a pretrained ResNet34 model using transfer learning.

Training includes:

- Frozen backbone
- Fine-tuned classifier
- AdamW optimization
- Label smoothing
- Early stopping
- Learning-rate scheduling

In [6]:
for fold,(train_idx,val_idx) in enumerate(
    skf.split(np.arange(len(dataset)),labels),1):

    print("="*60)
    print(f"Fold {fold}")
    print("="*60)

    train_dataset = copy.deepcopy(dataset)
    train_dataset.transform = train_transform

    val_dataset = copy.deepcopy(dataset)
    val_dataset.transform = test_transform

    train_subset = Subset(train_dataset,train_idx)
    val_subset = Subset(val_dataset,val_idx)

    train_loader = DataLoader(
        train_subset,
        batch_size=8,
        shuffle=True
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=8,
        shuffle=False
    )

    # ---------------- MODEL ----------------

    weights = models.ResNet34_Weights.DEFAULT

    model = models.resnet34(weights=weights)

    # Freeze backbone
    for param in model.parameters():
        param.requires_grad = False

    # Fine tune last two stages
    for param in model.layer3.parameters():
        param.requires_grad = True

    for param in model.layer4.parameters():
        param.requires_grad = True

    model.fc = nn.Linear(512,2)

    model = model.to(device)

    # ---------------- LOSS ----------------

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )

    # ---------------- OPTIMIZER ----------------

    optimizer = optim.AdamW(
        filter(lambda p:p.requires_grad,
               model.parameters()),
        lr=5e-5,
        weight_decay=1e-4
    )

    # ---------------- SCHEDULER ----------------

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    best_acc = 0

    counter = 0

    best_model = copy.deepcopy(model.state_dict())

    # ---------------- TRAIN ----------------

    for epoch in range(EPOCHS):

        loss,train_acc = train_one_epoch(
            model,
            train_loader
        )

        val_acc,preds,targets = evaluate(
            model,
            val_loader
        )

        scheduler.step(val_acc)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {loss:.4f} | "
            f"Train: {train_acc:.4f} | "
            f"Val: {val_acc:.4f}"
        )

        if val_acc > best_acc:

            best_acc = val_acc

            best_model = copy.deepcopy(
                model.state_dict()
            )

            counter = 0

        else:

            counter += 1

        if counter >= PATIENCE:

            print("Early Stopping...")
            break

    # ---------------- LOAD BEST ----------------

    model.load_state_dict(best_model)

    val_acc,preds,targets = evaluate(
        model,
        val_loader
    )

    precision = precision_score(targets,preds)

    recall = recall_score(targets,preds)

    f1 = f1_score(targets,preds)

    print(f"\nBest Validation Accuracy : {val_acc:.4f}")

    if val_acc > best_fold_accuracy:

        best_fold_accuracy = val_acc

        best_fold = fold

        torch.save(
            model.state_dict(),
            "../models/best_resnet34_detector.pth"
        )

        print("Best model saved!")

    fold_results.append({

        "accuracy":val_acc,
        "precision":precision,
        "recall":recall,
        "f1":f1

    })

Fold 1
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /Users/riyagarg/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100.0%


Epoch 01 | Loss: 0.6486 | Train: 0.5926 | Val: 0.6667
Epoch 02 | Loss: 0.3807 | Train: 0.9136 | Val: 0.7143
Epoch 03 | Loss: 0.3271 | Train: 0.9630 | Val: 0.8095
Epoch 04 | Loss: 0.2994 | Train: 0.9753 | Val: 0.8095
Epoch 05 | Loss: 0.2697 | Train: 1.0000 | Val: 0.8095
Epoch 06 | Loss: 0.2768 | Train: 1.0000 | Val: 0.8095
Epoch 07 | Loss: 0.2776 | Train: 0.9753 | Val: 0.8571
Epoch 08 | Loss: 0.2641 | Train: 0.9877 | Val: 0.7619
Epoch 09 | Loss: 0.2976 | Train: 0.9877 | Val: 0.8095
Epoch 10 | Loss: 0.2855 | Train: 0.9630 | Val: 0.7619
Epoch 11 | Loss: 0.2718 | Train: 0.9877 | Val: 0.8095
Epoch 12 | Loss: 0.2763 | Train: 0.9877 | Val: 0.8095
Early Stopping...

Best Validation Accuracy : 0.8571
Best model saved!
Fold 2
Epoch 01 | Loss: 0.8023 | Train: 0.5062 | Val: 0.6667
Epoch 02 | Loss: 0.4266 | Train: 0.8889 | Val: 0.6667
Epoch 03 | Loss: 0.3549 | Train: 0.9259 | Val: 0.8095
Epoch 04 | Loss: 0.2887 | Train: 0.9630 | Val: 0.7619
Epoch 05 | Loss: 0.2988 | Train: 0.9753 | Val: 0.6667
Epoc

## Aggregate Cross-Validation Results

Calculate the mean and standard deviation of all evaluation metrics across the five folds.

In [7]:
acc = [x["accuracy"] for x in fold_results]
prec = [x["precision"] for x in fold_results]
rec = [x["recall"] for x in fold_results]
f1 = [x["f1"] for x in fold_results]

print("\n"+"="*60)
print("Cross Validation Results")
print("="*60)

print("Accuracy :",acc)
print()

print("Precision :",prec)
print()

print("Recall :",rec)
print()

print("F1 :",f1)
print()

print(f"Mean Accuracy : {np.mean(acc):.4f}")
print(f"Std Accuracy  : {np.std(acc):.4f}")
print()

print(f"Mean Precision : {np.mean(prec):.4f}")
print(f"Mean Recall    : {np.mean(rec):.4f}")
print(f"Mean F1 Score  : {np.mean(f1):.4f}")

print("\n"+"="*60)
print(f"Best Fold : {best_fold}")
print(f"Best Accuracy : {best_fold_accuracy:.4f}")
print("../models/best_resnet34_detector.pth")
print("="*60)


Cross Validation Results
Accuracy : [0.8571428571428571, 0.8095238095238095, 0.85, 0.95, 0.8]

Precision : [0.8181818181818182, 0.8181818181818182, 0.8181818181818182, 0.9090909090909091, 0.75]

Recall : [0.9, 0.8181818181818182, 0.9, 1.0, 0.9]

F1 : [0.8571428571428571, 0.8181818181818182, 0.8571428571428571, 0.9523809523809523, 0.8181818181818182]

Mean Accuracy : 0.8533
Std Accuracy  : 0.0532

Mean Precision : 0.8227
Mean Recall    : 0.9036
Mean F1 Score  : 0.8606

Best Fold : 4
Best Accuracy : 0.9500
../models/best_resnet34_detector.pth


# Summary

This notebook evaluated ResNet34 using Stratified 5-Fold Cross Validation.

Among all evaluated transfer learning models, ResNet34 achieved the highest overall balance between Accuracy, Precision, Recall, and F1 Score while maintaining fast inference speed.

The model achieved a mean cross-validation accuracy of **85.33%** and a best-fold accuracy of **95.00%**, making it the final architecture selected for deployment in this project.